## 1. CONFIGURATION

**Projet :** Detection de fraude Mobile Money.
**Donnees :** 500 000 transactions synthetiques generees avec 5 types de fraude (account_takeover, stolen_card, mule_network, synthetic_id, smishing) et un taux de 0.15%.
**Pipeline :** 4 modeles sklearn (Isolation Forest, Random Forest, Logistic Regression, VotingClassifier) avec split temporel strict anti-leakage (train < 2025-07-01, test >= 2025-07-01).

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
plt.style.use('seaborn-v0_8-whitegrid')
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import joblib, json, os

## 2. CHARGEMENT DES DONNEES

Chargement des fichiers CSV depuis `data/sample/` (régénérables avec `python -m src.data.generator`). Les donnees synthetiques contiennent les transactions Mobile Money et les comptes associes pour les 8 pays UEMOA.

In [ ]:
tx = pd.read_csv('../data/sample/transactions.csv', parse_dates=['timestamp'])
acc = pd.read_csv('../data/sample/accounts.csv')
print(f"Transactions: {tx.shape[0]:,} | Fraudes: {tx.is_fraud.sum():,}")
print(f"Taux de fraude global: {tx.is_fraud.mean()*100:.3f}%")
print(f"Comptes: {acc.shape[0]:,}")
print(f"Periode: {tx['timestamp'].min().date()} -> {tx['timestamp'].max().date()}")

In [ ]:
print(f"Colonnes ({tx.shape[1]}): {list(tx.columns)}")
display(tx.head(3))
print("\nTypes de fraude:")
display(tx[tx['is_fraud'] == 1]['fraud_type'].value_counts().to_frame('Nombre'))
print("\nCanaux de transaction:")
display(tx['channel'].value_counts().to_frame('Nombre'))

## 3. EDA - ANALYSE EXPLORATOIRE

Analyse exploratoire complete : distribution de la cible, valeurs manquantes, distributions des montants, fraudes par heure/canal/operateur, correlations, PCA et DBSCAN.

In [ ]:
fc = tx['is_fraud'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(['Legitime', 'Fraude'], fc.values, color=['steelblue', 'crimson'])
axes[0].set_title('Distribution de la cible')
axes[1].pie(fc.values, labels=['Legitime', 'Fraude'], autopct='%1.3f%%',
            colors=['steelblue', 'crimson'], explode=(0, 0.1))
axes[1].set_title('Proportion fraude')
plt.tight_layout(); plt.show()

In [ ]:
import missingno as msno
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
msno.matrix(tx, ax=axes[0], sparkline=False)
axes[0].set_title('Matrice des valeurs manquantes')
msno.bar(tx, ax=axes[1])
axes[1].set_title('Completude par colonne')
plt.tight_layout(); plt.show()

In [ ]:
leg, fra = tx[tx['is_fraud'] == 0], tx[tx['is_fraud'] == 1]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(leg['amount'].sample(20000), bins=60, alpha=0.6, label='Legitime', color='steelblue')
axes[0].hist(fra['amount'], bins=30, alpha=0.7, label='Fraude', color='crimson')
axes[0].set_title('Montants bruts'); axes[0].legend()
axes[1].hist(np.log1p(leg['amount'].sample(20000)), bins=60, alpha=0.6, label='Legitime', color='steelblue')
axes[1].hist(np.log1p(fra['amount']), bins=30, alpha=0.7, label='Fraude', color='crimson')
axes[1].set_title('Montants (log1p)'); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
tx_hour = tx.copy(); tx_hour['hour'] = tx_hour['timestamp'].dt.hour
fraud_rate_h = tx_hour.groupby('hour')['is_fraud'].mean() * 100
fig, ax = plt.subplots(figsize=(10, 4))
fraud_rate_h.plot(kind='bar', color='crimson', alpha=0.7, ax=ax)
ax.set_title('Taux de fraude par heure (%)'); ax.set_xlabel('Heure')
ax.set_ylabel('Taux de fraude (%)')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fraud_ch = tx[tx['is_fraud'] == 1]['channel'].value_counts()
all_ch = tx['channel'].value_counts()
axes[0].bar(fraud_ch.index, fraud_ch.values, color='crimson')
axes[0].set_title('Fraudes par canal')
axes[1].pie(all_ch.values, labels=all_ch.index, autopct='%1.1f%%',
            colors=sns.color_palette('pastel'))
axes[1].set_title('Volume total par canal')
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
fraud_op = tx[tx['is_fraud'] == 1]['operator'].value_counts()
all_op = tx['operator'].value_counts()
(fraud_op / all_op * 100).sort_values(ascending=False).plot(
    kind='bar', color=['orange', 'crimson', 'steelblue'], ax=ax)
ax.set_title('Taux de fraude par operateur (%)'); ax.set_ylabel('%')
plt.tight_layout(); plt.show()

In [ ]:
corr_tx = tx.assign(
    amount_log=np.log1p(tx['amount']),
    hour=tx['timestamp'].dt.hour,
    dow=tx['timestamp'].dt.dayofweek,
    is_night=((tx['timestamp'].dt.hour >= 22) | (tx['timestamp'].dt.hour <= 5)).astype(int),
    is_weekend=(tx['timestamp'].dt.dayofweek >= 5).astype(int))
num_cols = ['amount', 'amount_log', 'is_fraud', 'hour', 'dow', 'is_night', 'is_weekend']
corr = corr_tx[num_cols].corr()
fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, ax=ax, linewidths=0.5)
ax.set_title('Matrice de correlation (variables numeriques)')
plt.tight_layout(); plt.show()

In [ ]:
sample = tx.sample(5000, random_state=42).copy()
sample['amount_log'] = np.log1p(sample['amount'])
sample['hour'] = sample['timestamp'].dt.hour
sample['fraud_label'] = sample['is_fraud'].map({0: 'Legitime', 1: 'Fraude'})
g = sns.pairplot(sample[['amount_log', 'hour', 'fraud_label']],
    hue='fraud_label', palette={'Legitime': 'steelblue', 'Fraude': 'crimson'}, height=3)
plt.suptitle('Pairplot: montant_log vs heure', y=1.02); plt.show()

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
samp = tx.sample(10000, random_state=42).copy()
samp['amount_log'] = np.log1p(samp['amount'])
samp['hour'] = samp['timestamp'].dt.hour; samp['dow'] = samp['timestamp'].dt.dayofweek
samp['is_night'] = ((samp['hour'] >= 22) | (samp['hour'] <= 5)).astype(int)
X_pca_in = StandardScaler().fit_transform(samp[['amount_log','hour','dow','is_night']].fillna(0))
pca = PCA(n_components=2); X_pca = pca.fit_transform(X_pca_in)
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=samp['is_fraud'], cmap='coolwarm', alpha=0.5, s=5)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.set_title('ACP - Projection 2D des transactions')
plt.colorbar(sc, label='Fraude'); plt.tight_layout(); plt.show()

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
samp = tx.sample(5000, random_state=42).copy()
samp['amount_log'] = np.log1p(samp['amount']); samp['hour'] = samp['timestamp'].dt.hour
X_db = StandardScaler().fit_transform(samp[['amount_log', 'hour']].fillna(0))
db = DBSCAN(eps=0.5, min_samples=10); clusters = db.fit_predict(X_db)
n_c = len(set(clusters) - {-1}); n_noise = (clusters == -1).sum()

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['gray' if c == -1 else plt.cm.tab10(c % 10) for c in clusters]
ax.scatter(samp['amount_log'], samp['hour'], c=colors, alpha=0.6, s=10)
ax.set_xlabel('Montant (log)'); ax.set_ylabel('Heure')
ax.set_title(f'DBSCAN: {n_c} clusters, {n_noise} bruits')
plt.tight_layout(); plt.show()

## 4. SYNTHESE EDA

**Constatations :** Le dataset est fortement desequilibre (0.15% de fraude). Les montants frauduleux sont significativement plus eleves, particulierement pour les types account_takeover et synthetic_id. La fraude est plus frequente la nuit (22h-5h) et les canaux mobile_app et ussd sont les plus touches. L'ACP montre un chevauchement important entre classes legitimes et frauduleuses, confirmant la difficulte de la tache. Le DBSCAN detecte des micro-clusters de points isoles correspondant potentiellement a des comportements frauduleux.

## 5. SPLIT TEMPOREL ANTI-LEAKAGE

**Regle absolue :** le split est temporel, jamais aleatoire. Toute transaction avant le 1er juillet 2025 appartient au train, apres au test. Cela evite le data leakage (fuite d'information du futur vers le passe) et simule un deploiement realiste.

In [ ]:
cutoff = pd.Timestamp('2025-07-01')
train_mask = tx['timestamp'] < cutoff
test_mask = tx['timestamp'] >= cutoff
tx_train = tx[train_mask].copy(); tx_test = tx[test_mask].copy()
acc_train = acc[acc['account_id'].isin(tx_train['sender_id'])].copy()
print(f"Train: {len(tx_train):,} ({tx_train['is_fraud'].mean()*100:.3f}% fraude)")
print(f"Test:  {len(tx_test):,} ({tx_test['is_fraud'].mean()*100:.3f}% fraude)")

## 6. FEATURE ENGINEERING

Creation de 4 categories de features : temporelles (encodage cyclique heure, nuit/weekend), comportementales (rolling 24h sur les montants), profil emetteur (age compte, KYC, SIM swap) et encodage one-hot des variables categorielles. Chaque etape est detaillee ci-dessous avant l'assemblage final via `src/features/engineering.py`.

In [ ]:
demo = tx_train.merge(acc_train, left_on='sender_id', right_on='account_id',
                      how='left', suffixes=('', '_acc'))
h = demo['timestamp'].dt.hour
demo['hour_sin'] = np.sin(2 * np.pi * h / 24); demo['hour_cos'] = np.cos(2 * np.pi * h / 24)
demo['is_night'] = ((h >= 22) | (h <= 5)).astype(int)
demo['is_weekend'] = (demo['timestamp'].dt.dayofweek >= 5).astype(int)
display(demo[['timestamp', 'hour_sin', 'hour_cos', 'is_night', 'is_weekend']].head(3))

In [ ]:
# Demonstration: fenetre glissante 24h (via merge, closed='left')
demo['amount_log'] = np.log1p(demo['amount'])
idx = demo.set_index('timestamp').sort_index()
g = idx.groupby('sender_id')['amount']; w = g.rolling('24h', closed='left')
demo = demo.merge(w.count().reset_index(name='tx_count_24h'), on=['sender_id','timestamp'], how='left')
demo = demo.merge(w.sum().reset_index(name='amount_sum_24h'), on=['sender_id','timestamp'], how='left')
demo[['tx_count_24h','amount_sum_24h']] = demo[['tx_count_24h','amount_sum_24h']].fillna(0)
display(demo[['amount','amount_log','tx_count_24h','amount_sum_24h']].head(3))

In [ ]:
# Demonstration: age du compte, KYC, SIM swap, ratio montant vs moyenne
sender_first = demo.groupby('sender_id')['timestamp'].min()
demo['sender_account_age'] = (demo['timestamp'] - demo['sender_id'].map(sender_first)).dt.days
demo['sender_is_kyc_verified'] = demo['is_kyc_verified'].fillna(0).astype(int)
demo['sender_sim_swap_recent'] = demo['sim_swap_recent'].fillna(0).astype(int)
sender_avg = demo.groupby('sender_id')['amount'].transform('mean')
demo['amount_vs_avg'] = demo['amount'] / (sender_avg + 1)
display(demo[['sender_account_age', 'sender_is_kyc_verified', 'amount_vs_avg']].head(3))

In [ ]:
# Demonstration: one-hot encoding des variables categorielles
demo['day_of_week'] = demo['timestamp'].dt.dayofweek
demo_oh = pd.get_dummies(demo, columns=[
    'transaction_type', 'channel', 'operator', 'amount_category', 'day_of_week'
], drop_first=True)
cat_cols = [c for c in demo_oh.columns if any(
    p in c for p in ['transaction_type_', 'channel_', 'operator_'])]
print(f"{len(cat_cols)} colonnes one-hot creees"); display(demo_oh[cat_cols[:6]].head(2))

In [ ]:
from src.features.engineering import engineer_features_train, engineer_features_test

X_train, y_train, train_params = engineer_features_train(tx_train, acc_train)
X_test, y_test = engineer_features_test(tx_test, acc, train_params)
print(f"X_train: {X_train.shape} | y_train: {len(y_train):,} (fraude={y_train.mean()*100:.3f}%)")
print(f"X_test:  {X_test.shape} | y_test:  {len(y_test):,} (fraude={y_test.mean()*100:.3f}%)")
print(f"Features ({X_train.shape[1]}): {list(X_train.columns[:8])}...")

## 7. MODELISATION

Entrainement de 4 modeles via `src/models/train.py` : Isolation Forest (non supervise), Random Forest, Logistic Regression et VotingClassifier (RF+LR). Chaque modele est evalue avec AUC-ROC, AUC-PR, Recall@FPR=1% et Lift@5%.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print(f"X_train_scaled: mean={X_train_s.mean():.6f}, std={X_train_s.std():.3f}")
print(f"X_test_scaled:  mean={X_test_s.mean():.6f}, std={X_test_s.std():.3f}")

In [ ]:
from src.models.train import train_isolation_forest, evaluate_model

if_model = train_isolation_forest(X_train)
if_metrics = evaluate_model(if_model, X_test, y_test)
print(f"Isolation Forest | AUC-PR: {if_metrics['auc_pr']:.4f} | AUC-ROC: {if_metrics['auc_roc']:.4f}")
print(f"Recall@FPR=1%: {if_metrics['recall_at_fpr_1pct']:.4f} | Lift@5%: {if_metrics['lift_at_5pct']:.4f}")

In [ ]:
from src.models.train import train_random_forest

rf_model = train_random_forest(X_train, y_train)
rf_metrics = evaluate_model(rf_model, X_test, y_test)
print(f"Random Forest   | AUC-PR: {rf_metrics['auc_pr']:.4f} | AUC-ROC: {rf_metrics['auc_roc']:.4f}")
print(f"Recall@FPR=1%: {rf_metrics['recall_at_fpr_1pct']:.4f} | Lift@5%: {rf_metrics['lift_at_5pct']:.4f}")

In [ ]:
from src.models.train import train_logistic_regression

lr_model = train_logistic_regression(X_train, y_train)
lr_metrics = evaluate_model(lr_model, X_test, y_test)
print(f"Logistic Reg     | AUC-PR: {lr_metrics['auc_pr']:.4f} | AUC-ROC: {lr_metrics['auc_roc']:.4f}")
print(f"Recall@FPR=1%: {lr_metrics['recall_at_fpr_1pct']:.4f} | Lift@5%: {lr_metrics['lift_at_5pct']:.4f}")

In [ ]:
from src.models.train import train_voting_ensemble

vote_model = train_voting_ensemble(X_train, y_train)
vote_metrics = evaluate_model(vote_model, X_test, y_test)
print(f"VotingClassifier | AUC-PR: {vote_metrics['auc_pr']:.4f} | AUC-ROC: {vote_metrics['auc_roc']:.4f}")
print(f"Recall@FPR=1%: {vote_metrics['recall_at_fpr_1pct']:.4f} | Lift@5%: {vote_metrics['lift_at_5pct']:.4f}")

In [ ]:
from src.evaluation.metrics import all_fraud_metrics

vote_scores = vote_model.predict_proba(X_test)[:, 1]
full_m = all_fraud_metrics(y_test, vote_scores)
for k in ['auc_roc', 'auc_pr', 'precision', 'recall', 'f1',
          'precision_at_recall_80', 'recall_at_fpr_1pct', 'lift_at_5pct']:
    print(f"  {k:25s}: {full_m[k]:.4f}")
print(f"  Taux de fraude test: {full_m['fraud_rate']*100:.3f}%")

In [ ]:
models_dict = {'IsolationForest': if_metrics, 'RandomForest': rf_metrics,
               'LogisticRegression': lr_metrics, 'VotingClassifier': vote_metrics}
df_cmp = pd.DataFrame(models_dict).T
display(df_cmp[['auc_roc', 'auc_pr', 'recall_at_fpr_1pct', 'lift_at_5pct']]
        .round(4).style.highlight_max(axis=0, color='lightgreen'))

## 8. ANALYSE COUT-BENEFICE

Evaluation economique du système de detection. Chaque faux positif coute 1 000 FCFA (blocage inutile), chaque vrai positif rapporte 50 000 FCFA (fraude evitée). Les faux negatifs coutent 50 000 FCFA (fraude non detectée).

In [ ]:
from src.evaluation.metrics import cost_benefit_analysis

models_cb = {'RF': rf_model, 'LR': lr_model, 'Voting': vote_model}
cb = {n: cost_benefit_analysis(y_test, m.predict_proba(X_test)[:, 1])
      for n, m in models_cb.items()}

# ── BENEFICE NET (remplace l'ancien graphique cout/benefice brut trompeur) ──
# Un modele avec beaucoup de TP ET de FP/FN aura 2 grandes barres
# (cout eleve, benefice eleve) mais un net negatif. Ce graphique montre
# directement le net pour eviter cette illusion d'optique.
fig, ax = plt.subplots(figsize=(10, 6))
names = list(cb.keys())
x = np.arange(len(names))
nets = [cb[n]['net_benefit'] for n in names]

# Couleurs : bleu si positif, rouge si negatif
colors = ['#2e86de' if v >= 0 else '#ee5a24' for v in nets]
bars = ax.bar(x, nets, color=colors, edgecolor='white', linewidth=1.2, width=0.55)

# Ligne zero
ax.axhline(y=0, color='black', linewidth=1.5, linestyle='-')

# Annotations avec le montant net
for i, (bar, net) in enumerate(zip(bars, nets)):
    offset = max(abs(v) for v in nets) * 0.03
    y_pos = bar.get_height() + (offset if net >= 0 else -offset)
    va = 'bottom' if net >= 0 else 'top'
    ax.text(bar.get_x() + bar.get_width() / 2, y_pos,
            f'{net:+,.0f} F', ha='center', va=va,
            fontweight='bold', fontsize=12, color=colors[i])

ax.set_xticks(x)
ax.set_xticklabels(['Random Forest', 'Logistic Regression', 'Voting Classifier'],
                   fontsize=11)
ax.set_ylabel('Benefice Net (FCFA)', fontsize=12)
ax.set_title('Analyse Cout-Benefice — Benefice Net par Modele\n'
             '(bleu = rentable, rouge = perte)', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:+,.0f}'))
plt.tight_layout()
plt.show()

for n in names:
    print(f"{n:10s} | Net: {cb[n]['net_benefit']:+,.0f}F | "
          f"ROI: {cb[n]['roi']:+.1f}% | "
          f"TP={cb[n]['true_positives']} FP={cb[n]['false_positives']} "
          f"FN={cb[n]['false_negatives']}")

## 9. RESULTATS FINAUX

RF et VotingClassifier offrent le meilleur compromis detection/cout avec un ROI > 500%. La Regression Logistique, malgre de bonnes metriques de classification (AUC-ROC 0.90, Lift 14.6x), a un **benefice net negatif** (-7.8M FCFA, ROI -35%) : elle genere trop de faux positifs et de faux negatifs par rapport aux fraudes qu'elle detecte. Ce cas illustre pourquoi l'analyse cout-benefice est indispensable en complement des metriques de classification.

### Verification des objectifs

- **AUC-ROC > 0.90** : OK pour RF, LR et VotingClassifier
- **AUC-PR > 0.30** : OK pour Random Forest, Logistic Regression et VotingClassifier (amelioration significative vs hasard = 0.15%)
- **Recall@FPR=1% > 10%** : OK, detection efficace avec tres peu de faux positifs
- **Lift@5% > 10x** : OK pour RF, LR et VotingClassifier
- **ROI > 500%** : OK pour RF et Voting uniquement. ⚠️ La Regression Logistique a un ROI **negatif (-35%)** malgre un AUC-ROC de 0.90 et un Lift de 14.6x : le cout des faux negatifs + faux positifs depasse le benefice des vrais positifs. Le graphique cout/benefice brut etait trompeur (2 grandes barres = illusion de performance), d'ou le remplacement par le **benefice net**.

In [ ]:
recap = pd.DataFrame({
    'AUC-ROC': [if_metrics['auc_roc'], rf_metrics['auc_roc'], lr_metrics['auc_roc'], vote_metrics['auc_roc']],
    'AUC-PR': [if_metrics['auc_pr'], rf_metrics['auc_pr'], lr_metrics['auc_pr'], vote_metrics['auc_pr']],
    'Recall@1%FPR': [if_metrics['recall_at_fpr_1pct'], rf_metrics['recall_at_fpr_1pct'], lr_metrics['recall_at_fpr_1pct'], vote_metrics['recall_at_fpr_1pct']],
    'Lift@5%': [if_metrics['lift_at_5pct'], rf_metrics['lift_at_5pct'], lr_metrics['lift_at_5pct'], vote_metrics['lift_at_5pct']]
}, index=['IsolationForest','RandomForest','LogisticRegression','VotingClassifier'])
display(recap.round(4).style.background_gradient(cmap='Blues'))

## 10. CONCLUSION

Ce notebook a presente un pipeline complet de detection de fraude Mobile Money, de l'analyse exploratoire a l'evaluation economique, en respectant des principes stricts d'anti-leakage temporel. Le VotingClassifier (Random Forest + Logistic Regression) est deployable en production avec un ROI de plus de 500%.

**Prochaines etapes :** deploiement API via FastAPI (`api/`), monitoring continu avec MLflow et Prometheus (`monitoring/`), dashboard interactif (`dashboard-react/`), et tests A/B pour mesurer l'impact metier reel.